In [0]:
%run "/Users/prdconfigsalesrep@gmail.com/databricks-project/config/env_config"

In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env").strip()

config = get_env_config(env)

catalog = config["catalog"]
schema = config["schema"]

In [0]:
from pyspark.sql.functions import upper, trim, col

In [0]:
customers_df = spark.read.table(f"{catalog}.{schema}.bronze_customers")
orders_df = spark.read.table(f"{catalog}.{schema}.bronze_orders")
display(customers_df)
display(orders_df)

In [0]:
customers_silver = customers_df \
    .withColumn("name", trim(col("name"))) \
    .withColumn("city", upper(trim(col("city"))))

In [0]:
table_name = f"{catalog}.{schema}.silver_customers"

customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

In [0]:
silver_path = f"/Volumes/{catalog}/{schema}/silver_volume/"

In [0]:
customers_silver.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(silver_path)

   

In [0]:
orders_df = spark.read.table(f"{catalog}.{schema}.bronze_orders")


In [0]:
orders_df.show()

In [0]:
orders_silver = orders_df \
    .withColumn("amount", col("amount").cast("int")) \
    .dropna(subset=["customer_id", "amount"])

In [0]:
orders_silver.show()

In [0]:
table_name = f"{catalog}.{schema}.silver_orders"

orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)